# Import all necessary libraries and data sets

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import seaborn as sns
%matplotlib inline

#### In the next step I import all tables for a certain quarter (Q3/2023) and city (Munich) to understand the data base.

In [4]:
#def import_data(import_folder)
m_2309_lst = pd.read_csv('input/munich_2309/listings.csv')
m_2309_ngb = pd.read_csv('input/munich_2309/neighbourhoods.csv')
m_2309_rvw = pd.read_csv('input/munich_2309/reviews.csv')

m_2309_cal = pd.read_csv('input/munich_2309/calendar.csv.gz', compression='gzip')
m_2309_lst2 = pd.read_csv('input/munich_2309/listings.csv.gz', compression='gzip')
m_2309_rvw2 = pd.read_csv('input/munich_2309/reviews.csv.gz', compression='gzip')

m_2309_geo = gpd.read_file('input/munich_2309/neighbourhoods.geojson')

df_list = [m_2309_lst, m_2309_lst2, m_2309_rvw, m_2309_rvw2, m_2309_cal, m_2309_ngb, m_2309_geo]

In [274]:
m_2309_lst.head(3)

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,97945,Rental unit in Munich · ★4.84 · 1 bedroom · 1 ...,517685,Angelika,NaN,Hadern,48.11492,11.48954,Entire home/apt,97,2,129,2019-10-03,0.88,2,89,0,NaN
1,114695,Rental unit in Munich · ★4.80 · 1 bedroom · 3 ...,581737,Stephan,NaN,Berg am Laim,48.12071,11.63758,Entire home/apt,111,1,60,2023-09-19,0.40,3,106,5,NaN
2,127383,Rental unit in Munich · ★4.83 · 1 bedroom · 1 ...,630556,Sonja,NaN,Maxvorstadt,48.15199,11.56482,Entire home/apt,120,3,108,2023-07-29,0.72,1,0,1,NaN


In [275]:
m_2309_lst2.head(3)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
1504,97945,https://www.airbnb.com/rooms/97945,20230922223227,2023-09-23,city scrape,Rental unit in Munich · ★4.84 · 1 bedroom · 1 ...,"<b>The space</b><br />We offer a modern, quiet...",We are living in a outskirt of Munich its call...,https://a0.muscache.com/pictures/2459996/10b4c...,517685,https://www.airbnb.com/users/show/517685,Angelika,2011-04-18,"Munich, Germany",Ich freue mich auf viele internationale Gäste!...,within a day,75%,0%,f,https://a0.muscache.com/im/users/517685/profil...,https://a0.muscache.com/im/users/517685/profil...,Hadern,2,3,"['email', 'phone']",t,t,"Munich, Bavaria, Germany",Hadern,NaN,48.11492,11.48954,Entire rental unit,Entire home/apt,2,NaN,1 bath,1.0,1.0,"[""Heating"", ""Paid dryer"", ""Oven"", ""Room-darken...",$97.00,2,90,2,2,90,90,2.0,90.0,NaN,t,1,1,1,89,2023-09-23,129,0,0,2011-10-03,2019-10-03,4.84,4.89,4.82,4.90,4.93,4.78,4.73,NaN,f,2,2,0,0,0.88
2190,114695,https://www.airbnb.com/rooms/114695,20230922223227,2023-09-23,city scrape,Rental unit in Munich · ★4.80 · 1 bedroom · 3 ...,<b>The space</b><br />It´s a quiet and sunny a...,NaN,https://a0.muscache.com/pictures/21571874/960e...,581737,https://www.airbnb.com/users/show/581737,Stephan,2011-05-12,"Munich, Germany",I am looking forward to meet interesting peopl...,within an hour,100%,86%,t,https://a0.muscache.com/im/users/581737/profil...,https://a0.muscache.com/im/users/581737/profil...,Berg am Laim,3,3,"['email', 'phone']",t,t,NaN,Berg am Laim,NaN,48.12071,11.63758,Entire rental unit,Entire home/apt,5,NaN,1 bath,1.0,3.0,"[""Heating"", ""Oven"", ""Smoke alarm"", ""Cooking ba...",$111.00,1,30,1,1,30,30,1.0,30.0,NaN,t,18,27,27,106,2023-09-23,60,5,1,2011-07-08,2023-09-19,4.80,4.75,4.97,4.97,5.00,4.58,4.60,NaN,f,3,3,0,0,0.40
6030,127383,https://www.airbnb.com/rooms/127383,20230922223227,2023-09-23,previous scrape,Rental unit in Munich · ★4.83 · 1 bedroom · 1 ...,<b>The space</b><br />My cosy apartment is loc...,NaN,https://a0.muscache.com/pictures/79238c11-bc61...,630556,https://www.airbnb.com/users/show/630556,Sonja,2011-05-26,"Munich, Germany","Hi, mein Name ist Sonja und ich freue mich net...",within an hour,100%,75%,f,https://a0.muscache.com/im/users/630556/profil...,https://a0.muscache.com/im/users/630556/profil...,Maxvorstadt,2,2,"['email', 'phone', 'work_email']",t,t,NaN,Maxvorstadt,NaN,48.15199,11.56482,Entire rental unit,Entire home/apt,4,NaN,1 bath,1.0,1.0,"[""Heating"", ""Breakfast"", ""Private patio or bal...",$120.00,3,14,3,3,14,14,3.0,14.0,NaN,t,0,0,0,0,2023-09-23,108,1,0,2011-06-04,2023-07-29,4.83,4.91,4.79,4.98,4.95,4.91,4.79,NaN,f,1,1,0,0,0.72


In [7]:
m_2309_rvw.head(3)

,listing_id,date
0,97945,2011-10-03
1,97945,2011-12-20
2,97945,2012-03-02


In [8]:
m_2309_rvw2.head(3)

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,97945,591337,2011-10-03,508190,Lauren,We very much enjoyed staying at Angelika's apa...
1,97945,796084,2011-12-20,341171,Marty & Evelyn,"Angelika was a very gracious host, and went ou..."
2,97945,964528,2012-03-02,1647967,Sue,The apartment was beautiful and Angelika provi...


In [9]:
m_2309_cal.head(3)

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,97945,2023-09-23,f,$120.00,$120.00,2,90
1,97945,2023-09-24,f,$120.00,$120.00,2,90
2,97945,2023-09-25,t,$120.00,$120.00,2,90


In [331]:
m_2309_ngb.head(3)
#m_2309_ngb.value_counts

,neighbourhood_group,neighbourhood
0,NaN,Allach-Untermenzing
1,NaN,Altstadt-Lehel
2,NaN,Aubing-Lochhausen-Langwied


In [11]:
m_2309_geo.head(3)

,neighbourhood,neighbourhood_group,geometry
0,Altstadt-Lehel,None,"MULTIPOLYGON (((11.59520 48.14170, 11.59500 48..."
1,Ludwigsvorstadt-Isarvorstadt,None,"MULTIPOLYGON (((11.55600 48.14080, 11.55930 48..."
2,Maxvorstadt,None,"MULTIPOLYGON (((11.58430 48.14420, 11.58310 48..."


In [343]:
for df in df_list:
    print("The dataframe {} has {} columns and {} rows".format([name for name, value in globals().items() if value is df][0], df.shape[1], df.shape[0]))

The dataframe m_2309_lst has 18 columns and 7336 rows
The dataframe m_2309_lst2 has 75 columns and 7336 rows
The dataframe m_2309_rvw has 2 columns and 160345 rows
The dataframe m_2309_rvw2 has 6 columns and 160345 rows
The dataframe m_2309_cal has 7 columns and 2677641 rows
The dataframe m_2309_ngb has 2 columns and 25 rows
The dataframe m_2309_geo has 3 columns and 25 rows


#### It seems like there are two versions of the files 'listings' and 'reviews'. Next, I want to check if they are similar and thus, if it possible to leave one of them out in the following steps. 
# Listings

In [344]:
# Check if column 'id' is unique
print(np.sum(m_2309_lst['id'].duplicated()))
print(np.sum(m_2309_lst2['id'].duplicated()))

0
0


In [277]:
def compare_columns(df_1, df_2, col):
    df1 = df_1.sort_values(by = df_1.columns[0]).reset_index(drop=True)[[col]].add_suffix('_1')
    df2 = df_2.sort_values(by = df_2.columns[0]).reset_index(drop=True)[[col]].add_suffix('_2')

    comp_2_cols = pd.concat([df1, df2], axis = 1)
    comparison = np.sum((comp_2_cols.iloc[:,0] != comp_2_cols.iloc[:,1]))
    print('There are {} unequal values when comparing column {}.'.format(comparison, col))


for column in list(m_2309_lst.columns):
    try:
        compare_columns(m_2309_lst, m_2309_lst2, str(column))
    except:
        continue

There are 0 unequal values when comparing column id.
There are 0 unequal values when comparing column name.
There are 0 unequal values when comparing column host_id.
There are 0 unequal values when comparing column host_name.
There are 7336 unequal values when comparing column neighbourhood.
There are 0 unequal values when comparing column latitude.
There are 0 unequal values when comparing column longitude.
There are 0 unequal values when comparing column room_type.
There are 7336 unequal values when comparing column price.
There are 0 unequal values when comparing column minimum_nights.
There are 0 unequal values when comparing column number_of_reviews.
There are 2027 unequal values when comparing column last_review.
There are 2027 unequal values when comparing column reviews_per_month.
There are 0 unequal values when comparing column calculated_host_listings_count.
There are 0 unequal values when comparing column availability_365.
There are 0 unequal values when comparing column num

#### Detailed check on the columns with deviations between the to datasets

In [328]:
df1 = m_2309_lst[['id','neighbourhood','price','last_review','reviews_per_month','license']].sort_values(by = 'id').reset_index(drop=True).add_suffix('_1')
df2 = m_2309_lst2[['id','neighbourhood', 'neighbourhood_cleansed','price','last_review','reviews_per_month','license']].sort_values(by = 'id').reset_index(drop=True).add_suffix('_2')
df1_df2 = pd.concat([df1, df2], axis = 1)
df1_df2


,id_1,neighbourhood_1,price_1,last_review_1,reviews_per_month_1,license_1,id_2,neighbourhood_2,neighbourhood_cleansed_2,price_2,last_review_2,reviews_per_month_2,license_2


In [322]:
'''NEIGHBOURHOOD'''

#Find all columns that contain the string 'neighbourhood' in m_2309_lst2 to check if there is a better matching column
m_2309_lst2.filter(like = 'neighbourhood')

#Check unequal values again for column 'neighbourhood_cleansed' as it seems better 
df1 = m_2309_lst.sort_values(by = m_2309_lst.columns[0]).reset_index(drop=True)[['neighbourhood']].add_suffix('_1')
df2 = m_2309_lst2.sort_values(by = m_2309_lst.columns[0]).reset_index(drop=True)[['neighbourhood_cleansed']].add_suffix('_2')

comp_2_cols = pd.concat([df1, df2], axis = 1)
comparison = np.sum((comp_2_cols.iloc[:,0] != comp_2_cols.iloc[:,1]))
print('There are {} unequal values when comparing the two columns.'.format(comparison))

# Use neighbourhood_cleansed

There are 0 unequal values when comparing the two columns.


In [323]:
'''PRICE'''

#Check format of column "price" in both datasets
print(m_2309_lst['price'].dtypes)
print(m_2309_lst2['price'].dtypes)

#Create new column in enriched data set with a new column that converts price from object to integer
df1 = m_2309_lst.sort_values(by = m_2309_lst.columns[0]).reset_index(drop=True)[['price']].add_suffix('_1')
df2 = m_2309_lst2.sort_values(by = m_2309_lst.columns[0]).reset_index(drop=True)[['price']].add_suffix('_2')
df2['price_adj_format_2'] = df2['price_2'].str.replace('$','').str.split('.').str[0].str.replace(',','').astype(int)

comp_2_cols = pd.concat([df1, df2], axis = 1)
comparison = np.sum(comp_2_cols.iloc[:,0] != comp_2_cols.iloc[:,2])
print('There are {} unequal values when comparing the two columns.'.format(comparison))

# Use pice from first dataset

int64
object
There are 0 unequal values when comparing the two columns.


In [340]:
#LAST REVIEW
#Here it seems like the differences stem from a different formatting of the NaN values. 
#Lets see if there is differences after filtering out missing Values. 

print(np.sum(df1_df2['last_review_1'].isna()))

filtered_df = df1_df2[(df1_df2['last_review_1'] != df1_df2['last_review_2']) & (df1_df2['last_review_1'].notna())]
filtered_df
# >> as expected, all differences between the datasets come from a different formatting of the NaN entries



2027


,id_1,neighbourhood_1,price_1,last_review_1,reviews_per_month_1,license_1,id_2,neighbourhood_2,neighbourhood_cleansed_2,price_2,last_review_2,reviews_per_month_2,license_2


In [341]:
#REVIEWS PER MONTH
print(np.sum(df1_df2['reviews_per_month_1'].isna()))

filtered_df = df1_df2[(df1_df2['reviews_per_month_1'] != df1_df2['reviews_per_month_2']) & (df1_df2['reviews_per_month_1'].notna())]
filtered_df
# >> as expected, all differences between the datasets come from a different formatting of the NaN entries


2027


,id_1,neighbourhood_1,price_1,last_review_1,reviews_per_month_1,license_1,id_2,neighbourhood_2,neighbourhood_cleansed_2,price_2,last_review_2,reviews_per_month_2,license_2


In [342]:
#LICENSE
print(np.sum(df1_df2['license_1'].isna()))

filtered_df = df1_df2[(df1_df2['license_1'] != df1_df2['license_2']) & (df1_df2['license_1'].notna())]
filtered_df
# >> as expected, all differences between the datasets come from a different formatting of the NaN entries


7315


,id_1,neighbourhood_1,price_1,last_review_1,reviews_per_month_1,license_1,id_2,neighbourhood_2,neighbourhood_cleansed_2,price_2,last_review_2,reviews_per_month_2,license_2


In [352]:
m_2309_lst_small = m_2309_lst
m_2309_lst = m_2309_lst2.drop('price',axis=1).merge(m_2309_lst_small[['id','price']], on='id', how='left')

print(np.sum(m_2309_lst_small['price']))
print(np.sum(m_2309_lst['price']))

1678379
1678379


In [356]:
m_2309_lst.describe()

,id,scrape_id,host_id,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bedrooms,beds,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,price
count,7.336000e+03,7.336000e+03,7.336000e+03,7336.000000,7336.000000,7336.000000,7336.000000,7336.000000,4843.000000,7261.000000,7336.000000,7336.000000,7336.000000,7336.000000,7.336000e+03,7.336000e+03,7336.000000,7.336000e+03,7336.000000,7336.000000,7336.000000,7336.000000,7336.000000,7336.000000,7336.000000,5318.000000,5308.000000,5308.000000,5308.000000,5308.000000,5308.000000,5308.000000,7336.000000,7336.000000,7336.000000,7336.000000,5309.000000,7336.000000
mean,4.487728e+17,2.023092e+13,1.714005e+08,4.043893,5.063931,48.139473,11.561735,2.767176,1.350196,1.639306,7.489368,392.296210,6.968239,8.135496,1.756879e+06,5.269679e+06,7.647628,2.711128e+06,9.073882,20.761859,33.860687,115.054526,21.857279,6.191521,0.592694,4.759579,4.804951,4.719164,4.844563,4.855595,4.783745,4.627924,2.936478,2.128135,0.760496,0.026445,1.010872,228.786668
std,4.277424e+17,2.023575e+00,1.722905e+08,22.423907,25.067307,0.025375,0.048528,1.624050,0.917730,1.342787,27.904934,432.854925,27.425471,28.765586,6.139424e+07,1.062509e+08,28.041129,7.153440e+07,10.067854,21.839150,34.203512,124.300011,66.261431,19.925587,1.580340,0.453984,0.377354,0.470023,0.355328,0.337103,0.347854,0.454651,8.286522,8.157963,2.129617,0.261835,1.639296,1155.948609
min,9.794500e+04,2.023092e+13,1.581000e+03,1.000000,1.000000,48.071440,11.388350,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.010000,9.000000
25%,3.358451e+07,2.023092e+13,2.445032e+07,1.000000,1.000000,48.122827,11.538430,2.000000,1.000000,1.000000,1.000000,26.000000,1.000000,1.000000,3.000000e+01,3.000000e+01,1.000000,3.000000e+01,0.000000,0.000000,0.000000,2.750000,0.000000,0.000000,0.000000,4.690000,4.750000,4.640000,4.830000,4.850000,4.710000,4.500000,1.000000,0.000000,0.000000,0.000000,0.190000,92.000000
50%,6.409723e+17,2.023092e+13,9.635626e+07,1.000000,2.000000,48.136970,11.563911,2.000000,1.000000,1.000000,2.000000,365.000000,2.000000,2.000000,3.650000e+02,3.650000e+02,2.000000,3.650000e+02,5.000000,11.000000,21.000000,66.000000,3.000000,1.000000,0.000000,4.900000,4.930000,4.890000,4.970000,5.000000,4.890000,4.730000,1.000000,1.000000,0.000000,0.000000,0.510000,150.000000
75%,8.919355e+17,2.023092e+13,2.977107e+08,2.000000,3.000000,48.155327,11.585353,4.000000,1.000000,2.000000,4.000000,365.000000,3.000000,4.000000,1.125000e+03,1.125000e+03,4.000000,1.125000e+03,18.000000,43.000000,70.000000,232.000000,14.000000,5.000000,1.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,2.000000,1.000000,1.000000,0.000000,1.120000,250.000000
max,9.856358e+17,2.023092e+13,5.382086e+08,553.000000,620.000000,48.228020,11.712699,16.000000,27.000000,50.000000,999.000000,1125.000000,999.000000,999.000000,2.147484e+09,2.147484e+09,999.000000,2.147484e+09,30.000000,60.000000,90.000000,365.000000,1650.000000,740.000000,44.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,76.000000,76.000000,26.000000,5.000000,56.120000,96274.000000


In [355]:
# Drop columns with only missing values
m_2309_lst = m_2309_lst.drop(['neighbourhood_group_cleansed','bathrooms','calendar_updated'], axis=1)

# Reviews
#### regarding the datasets m_2309_rvw and m_2309_rvw2 i will move forward with m_2309_rvw2 as it contains a lot more columns that are relevant

In [357]:
m_2309_rvw = m_2309_rvw2
m_2309_rvw2.describe()

,listing_id,id,reviewer_id
count,1.603450e+05,1.603450e+05,1.603450e+05
mean,8.083320e+16,3.628587e+17,1.511716e+08
std,2.319025e+17,3.910768e+17,1.460039e+08
min,9.794500e+04,2.976850e+05,1.000000e+00
25%,5.702154e+06,3.207912e+08,3.217967e+07
50%,2.068520e+07,7.259416e+08,9.972980e+07
75%,4.045921e+07,7.454135e+17,2.300618e+08
max,9.825751e+17,9.861474e+17,5.378926e+08


# Calendar

In [367]:
m_2309_cal.describe()
m_2309_cal['available'].value_counts()
print(m_2309_cal.shape)

compare_price = np.sum(m_2309_cal['price'] != m_2309_cal['adjusted_price'])
print('There are {} unequal values when comparing the two columns.'.format(compare_price))
print('There are {} missing values in the price column'.format(np.sum(m_2309_cal['price'].isna())))

(2677641, 7)
There are 66238 unequal values when comparing the two columns.
There are 0 missing values in the price column


In [371]:
#Check for differences in the price columns

m_2309_cal[['price','adjusted_price']][m_2309_cal['price'] != m_2309_cal['adjusted_price']]

#the price column seems to differ from the adjusted price column for a reason >> keep both

,price,adjusted_price
3650,$85.00,$78.00
3651,$85.00,$78.00
3652,$85.00,$78.00
17885,$58.00,$55.00
17886,$58.00,$55.00
...,...,...
2660121,$120.00,$84.00
2660122,$120.00,$84.00
2660123,$120.00,$84.00
2669611,$45.00,$36.00


# Neighbourhoods

In [376]:
m_2309_ngb
m_2309_ngb['neighbourhood_group'].value_counts()

# dataset will be dropped from the analysis

Series([], Name: count, dtype: int64)

# Neighbourhoods Geo

In [388]:
m_2309_geo
m_2309_geo['neighbourhood_group'].value_counts()
m_2309_geo.drop('neighbourhood_group', axis=1)

,neighbourhood,geometry
0,Altstadt-Lehel,"MULTIPOLYGON (((11.59520 48.14170, 11.59500 48..."
1,Ludwigsvorstadt-Isarvorstadt,"MULTIPOLYGON (((11.55600 48.14080, 11.55930 48..."
2,Maxvorstadt,"MULTIPOLYGON (((11.58430 48.14420, 11.58310 48..."
3,Schwabing-West,"MULTIPOLYGON (((11.58170 48.17630, 11.58320 48..."
4,Au-Haidhausen,"MULTIPOLYGON (((11.59560 48.14050, 11.59590 48..."
5,Sendling,"MULTIPOLYGON (((11.55290 48.10600, 11.55180 48..."
6,Sendling-Westpark,"MULTIPOLYGON (((11.53790 48.12650, 11.53770 48..."
7,Schwanthalerhöhe,"MULTIPOLYGON (((11.53460 48.14250, 11.53880 48..."
8,Neuhausen-Nymphenburg,"MULTIPOLYGON (((11.54810 48.17080, 11.55010 48..."
9,Moosach,"MULTIPOLYGON (((11.53300 48.18870, 11.53470 48..."


# Join relevant datasets

In [401]:
m_2309_geo.rename(columns={'neighbourhood': 'neighbourhood_cleansed'}, inplace=True)

m_2309_lst = m_2309_lst.merge(m_2309_geo[['neighbourhood_cleansed','geometry']], on ='neighbourhood_cleansed', how='left')
m_2309_lst

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_x,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,amenities,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,price,neighbourhood_y,geometry_x,neighbourhood,geometry_y,geometry
0,97945,https://www.airbnb.com/rooms/97945,20230922223227,2023-09-23,city scrape,Rental unit in Munich · ★4.84 · 1 bedroom · 1 ...,"<b>The space</b><br />We offer a modern, quiet...",We are living in a outskirt of Munich its call...,https://a0.muscache.com/pictures/2459996/10b4c...,517685,https://www.airbnb.com/users/show/517685,Angelika,2011-04-18,"Munich, Germany",Ich freue mich auf viele internationale Gäste!...,within a day,75%,0%,f,https://a0.muscache.com/im/users/517685/profil...,https://a0.muscache.com/im/users/517685/profil...,Hadern,2,3,"['email', 'phone']",t,t,"Munich, Bavaria, Germany",Hadern,48.114920,11.489540,Entire rental unit,Entire home/apt,2,1 bath,1.0,1.0,"[""Heating"", ""Paid dryer"", ""Oven"", ""Room-darken...",2,90,2,2,90,90,2.0,90.0,t,1,1,1,89,2023-09-23,129,0,0,2011-10-03,2019-10-03,4.84,4.89,4.82,4.90,4.93,4.78,4.73,NaN,f,2,2,0,0,0.88,97,Hadern,"MULTIPOLYGON (((11.47800 48.13050, 11.48300 48...",Hadern,"MULTIPOLYGON (((11.47800 48.13050, 11.48300 48...","MULTIPOLYGON (((11.47800 48.13050, 11.48300 48..."
1,114695,https://www.airbnb.com/rooms/114695,20230922223227,2023-09-23,city scrape,Rental unit in Munich · ★4.80 · 1 bedroom · 3 ...,<b>The space</b><br />It´s a quiet and sunny a...,NaN,https://a0.muscache.com/pictures/21571874/960e...,581737,https://www.airbnb.com/users/show/581737,Stephan,2011-05-12,"Munich, Germany",I am looking forward to meet interesting peopl...,within an hour,100%,86%,t,https://a0.muscache.com/im/users/581737/profil...,https://a0.muscache.com/im/users/581737/profil...,Berg am Laim,3,3,"['email', 'phone']",t,t,NaN,Berg am Laim,48.120710,11.637580,Entire rental unit,Entire home/apt,5,1 bath,1.0,3.0,"[""Heating"", ""Oven"", ""Smoke alarm"", ""Cooking ba...",1,30,1,1,30,30,1.0,30.0,t,18,27,27,106,2023-09-23,60,5,1,2011-07-08,2023-09-19,4.80,4.75,4.97,4.97,5.00,4.58,4.60,NaN,f,3,3,0,0,0.40,111,Berg am Laim,"MULTIPOLYGON (((11.60220 48.12390, 11.60360 48...",Berg am Laim,"MULTIPOLYGON (((11.60220 48.12390, 11.60360 48...","MULTIPOLYGON (((11.60220 48.12390, 11.60360 48..."
2,127383,https://www.airbnb.com/rooms/127383,20230922223227,2023-09-23,previous scrape,Rental unit in Munich · ★4.83 · 1 bedroom · 1 ...,<b>The space</b><br />My cosy apartment is loc...,NaN,https://a0.muscache.com/pictures/79238c11-bc61...,630556,https://www.airbnb.com/users/show/630556,Sonja,2011-05-26,"Munich, Germany","Hi, mein Name ist Sonja und ich freue mich net...",within an hour,100%,75%,f,https://a0.muscache.com/im/users/630556/profil...,https://a0.muscache.com/im/users/630556/profil...,Maxvorstadt,2,2,"['email', 'phone', 'work_email']",t,t,NaN,Maxvorstadt,48.151990,11.564820,Entire rental unit,Enti

## Final datasets for further analysis

### 1. m_2309_lst
### 2. m_2309_rvw
### 3. m_2309_cal

#### Drop: m_2309_lst (small version), m_2309_rvw (small version), m_2309_geo (joined), m_2309_ngb (no value added)